In [1]:
# 导入读取数据和计算所需的库。
import numpy as np
import pandas as pd

In [2]:
# 原始 Parquet 未随交付包分发；运行前把 data_file 改成本地分片路径。
from pathlib import Path

data_file = Path("YOUR_PARQUET_FILE.parquet")
field_file = Path("../自动化实现/字段说明.xlsx")
rules_file = Path("../自动化实现/数据质量规则.xlsx")

# 先用一个分片检查流程，后续再循环处理全部分片。
df = pd.read_parquet(data_file)
df.shape


(248139, 44)

In [3]:
# 按字段说明表统一筛选分析字段，避免在代码中反复手写字段名。
def select_analysis_fields(table, field_description):
    """根据字段说明表筛选分析层字段，并返回独立副本。

    Parameters
    ----------
    table : pandas.DataFrame
        待筛选的原始数据表。
    field_description : pandas.DataFrame
        字段说明表，必须包含“字段名”和“分析层处理”两列。
        “分析层处理”不等于“删除”的字段会被保留。

    Returns
    -------
    pandas.DataFrame
        只包含保留字段的数据表，字段顺序与字段说明表一致。

    Raises
    ------
    ValueError
        字段说明表要求保留的字段在原始数据中不存在时抛出。
    """
    # 字段说明表是字段取舍的唯一入口。
    selected = field_description["分析层处理"] != "删除"
    selected_fields = field_description.loc[selected, "字段名"].tolist()

    # 先检查字段是否齐全，避免筛选后才发现数据源缺列。
    missing_fields = []
    for field in selected_fields:
        if field not in table.columns:
            missing_fields.append(field)

    if len(missing_fields) > 0:
        raise ValueError(f"原始数据缺少字段：{missing_fields}")

    return table.loc[:, selected_fields].copy()


# 读取字段取舍结果，生成当前分析使用的工单表。
field_description = pd.read_excel(field_file, sheet_name="字段说明")
work_order = select_analysis_fields(df, field_description)
work_order.shape

(248139, 23)

In [4]:
# 查看筛选后各字段的数据类型和非空数量。
work_order.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 248139 entries, 0 to 248138
Data columns (total 23 columns):
 #   Column                          Non-Null Count   Dtype 
---  ------                          --------------   ----- 
 0   unique_key                      248139 non-null  object
 1   created_date                    248139 non-null  object
 2   closed_date                     243530 non-null  object
 3   agency                          248139 non-null  object
 4   agency_name                     248139 non-null  object
 5   complaint_type                  248139 non-null  object
 6   descriptor                      244214 non-null  object
 7   descriptor_2                    95227 non-null   object
 8   status                          248139 non-null  object
 9   due_date                        1169 non-null    object
 10  resolution_action_updated_date  247372 non-null  object
 11  open_data_channel_type          248139 non-null  object
 12  location_type                 

In [5]:
# 随机抽取15条记录，检查字段内容是否符合预期。
work_order.sample(15)

,unique_key,created_date,closed_date,agency,agency_name,complaint_type,descriptor,descriptor_2,status,due_date,...,address_type,city,borough,facility_type,park_facility_name,park_borough,vehicle_type,taxi_company_borough,bridge_highway_name,road_ramp
61654,62426515,2024-09-13T02:21:17.000,2024-09-13T03:22:41.000,NYPD,New York City Police Department,Noise - Residential,Loud Music/Party,None,Closed,None,...,ADDRESS,BROOKLYN,BROOKLYN,None,Unspecified,BROOKLYN,None,None,None,None
1127,62356595,2024-09-07T03:01:17.000,2024-09-07T03:39:46.000,NYPD,New York City Police Department,Noise - Street/Sidewalk,Loud Music/Party,None,Closed,None,...,ADDRESS,NEW YORK,MANHATTAN,None,Unspecified,MANHATTAN,None,None,None,None
190138,62558432,2024-09-24T15:25:38.000,2024-10-07T05:03:15.000,HPD,Department of Housing Preservation and Develop...,UNSANITARY CONDITION,PESTS,ROACHES,Closed,None,...,ADDRESS,BROOKLYN,BROOKLYN,None,Unspecified,BROOKLYN,None,None,None,None
9241,62365419,2024-09-07T23:14:28.000,2024-09-08T01:37:47.000,NYPD,New York City Police Department,Noise - Residential,Loud Music/Party,None,Closed,None,...,ADDRESS,NEW YORK,MANHATTAN,None,Unspecified,MANHATTAN,None,None,None,None
130011,62491225,2024-09-18T23:18:32.000,2024-09-19T00:27:17.000,NYPD,New York City Police Department,Illegal Parking,Commercial Overnight Parking,None,Closed,None,...,ADDRESS,BROOKLYN,BROOKLYN,None,Unspecified,BROOKLYN,None,None,None,None
155635,62515810,2024-09-21T16:37:29.000,2024-09-21T16:41:30.000,NYPD,New York City Police Department,Blocked Driveway,Partial Access,None,Closed,None,...,ADDRESS,BROOKLYN,BROOKLYN,None,Unspecified,BROOKLYN,None,None,None,None
200418,62570060,2024-09-25T17:13:10.000,2024-10-03T19:11:21.000,HPD,Department of Housing Preservation and Develop...,DOOR/WINDOW,DOOR,NEEDS KEY FOR EXIT,Closed,None,...,ADDRESS,BROOKLYN,BROOKLYN,None,Unspecified,BROOKLYN,None,None,None,None
41857,62400937,2024-09-11T08:13:54.000,2024-09-24T21:22:20.000,HPD,Department of Housing Preservation and Develop...,GENERAL,BELL/BUZZER/INTERCOM,BROKEN OR MISSING,Closed,None,...,ADDRESS,BRONX,BRONX,None,Unspecified,BRONX,None,None,None,None
114998,62482908,2024-09-17T12:45:19.000,2024-09-17T13:45:21.000,NYPD,New York City Police Department,Noise - Residential,Banging/Pounding,None,Closed,None,...,ADDRESS,BRONX,BRONX,None,Unspecified,BRONX,None,None,None,None
102306,62467212,2024-09-16T10:10:54.000,2024-10-12T12:26:06.000,DSNY,Department of Sanitation,Illegal Dumping,Removal Request,N/A,Closed,None,...,ADDRESS,JAMAICA,QUEENS,None,Unspecified,QUEENS,None,None,None,None


In [6]:
# 每种检查方式需要填写哪些参数。
REQUIRED_PARAMETERS = {
    "范围": ["最小值", "最大值"],
    "允许值": ["允许值"],
    "格式": ["正则表达式"],
    "不早于": ["对照字段"],
    "不晚于": ["对照字段"],
    "相同": ["对照字段"],
    "条件必填": ["对照字段"],
    "至少一个非空": ["对照字段"]
}
SUPPORTED_CHECKS = [
    "必填", "唯一", "日期", "数值", "整数"
] + list(REQUIRED_PARAMETERS)


# 先检查规则表能否执行，业务内容由规则编写者负责。
def validate_business_rules(rules, table, table_name):
    """检查业务规则表的基本结构，返回无法执行的规则。

    Parameters
    ----------
    rules : pandas.DataFrame
        从 Excel 读取的业务规则表。
    table : pandas.DataFrame
        准备接受检查的数据表。
    table_name : str
        当前数据表在规则表中的名称。

    Returns
    -------
    pandas.DataFrame
        规则填写错误；没有错误时返回空表。
    """
    columns = ["Excel行号", "规则编号", "错误原因"]
    errors = []
    required_columns = [
        "规则编号", "适用表", "字段名", "检查方式", "对照字段",
        "最小值", "最大值", "允许值", "正则表达式",
        "规则依据", "规则说明", "是否启用"
    ]

    for column in required_columns:
        if column not in rules.columns:
            errors.append([None, None, f"缺少必要列：{column}"])
    if len(errors) > 0:
        return pd.DataFrame(errors, columns=columns)

    duplicate_ids = rules["规则编号"].duplicated(keep=False)

    def is_blank(value):
        return pd.isna(value) or str(value).strip() == ""

    # 只检查程序运行必需的内容。
    for index in rules.index:
        rule = rules.loc[index]
        rule_id = rule["规则编号"]
        check = rule["检查方式"]
        row_errors = []

        if is_blank(rule_id):
            row_errors.append("规则编号不能为空")
        elif duplicate_ids.loc[index]:
            row_errors.append("规则编号重复")
        if rule["适用表"] != table_name:
            row_errors.append(f"适用表应为 {table_name}")
        if check not in SUPPORTED_CHECKS:
            row_errors.append(f"不支持的检查方式：{check}")
        if rule["是否启用"] not in ["是", "否"]:
            row_errors.append("是否启用只能填写‘是’或‘否’")

        if rule["是否启用"] == "是":
            field = rule["字段名"]
            if is_blank(field):
                row_errors.append("字段名不能为空")
            elif field not in table.columns:
                row_errors.append(f"数据表中不存在字段：{field}")

            for parameter in REQUIRED_PARAMETERS.get(check, []):
                if is_blank(rule[parameter]):
                    row_errors.append(f"{check}检查必须填写{parameter}")

            other_field = rule["对照字段"]
            if "对照字段" in REQUIRED_PARAMETERS.get(check, []):
                if not is_blank(other_field) and other_field not in table.columns:
                    row_errors.append(f"数据表中不存在对照字段：{other_field}")

            for field in ["规则依据", "规则说明"]:
                if is_blank(rule[field]):
                    row_errors.append(f"{field}不能为空")

        for reason in row_errors:
            errors.append([index + 2, rule_id, reason])

    return pd.DataFrame(errors, columns=columns)


all_business_rules = pd.read_excel(rules_file, sheet_name="业务规则")
rule_errors = validate_business_rules(
    all_business_rules,
    work_order,
    "work_order"
)

# rule_errors 非空时先修改规则表，再运行后面的审查代码。
business_rules = all_business_rules.loc[
    (all_business_rules["适用表"] == "work_order")
    & (all_business_rules["是否启用"] == "是")
].copy()

rule_errors

,Excel行号,规则编号,错误原因


In [7]:
# 把一条业务规则转换为适用范围掩码和错误掩码。
def check_rule(table, rule):
    """根据一条业务规则生成适用范围掩码和错误掩码。

    Parameters
    ----------
    table : pandas.DataFrame
        待审查的数据表。
    rule : pandas.Series
        数据质量规则表中的一行。必须提供“字段名”“检查方式”
        和“规则说明”；需要字段对比时还必须提供“对照字段”。

    Returns
    -------
    tuple
        返回 scope_mask、error_mask、error_type、error_reason。
        scope_mask 表示本规则需要检查的记录，error_mask 表示其中
        不符合规则的记录。

    Raises
    ------
    ValueError
        目标字段或对照字段不存在，或者“检查方式”不受支持时抛出。
    """
    column = rule["字段名"]
    check = rule["检查方式"]

    # 规则指定的字段不存在时停止，避免业务规则被静默跳过。
    if column not in table.columns:
        rule_id = rule["规则编号"]
        raise ValueError(f"规则 {rule_id} 的字段不存在：{column}")

    values = table[column]
    # 空字符串和只含空格的字符串也按空值处理。
    empty = values.isna() | (values.astype("string").str.strip() == "")
    all_records = pd.Series(True, index=table.index)

    # 先处理只检查一个字段的规则。
    if check == "必填":
        scope_mask = all_records
        error_mask = empty
        error_type = "完整性"

    elif check == "唯一":
        scope_mask = ~empty
        error_mask = scope_mask & values.duplicated(keep=False)
        error_type = "唯一性"

    elif check == "日期":
        # 临时转换只用于找出无法解析的非空值。
        converted = pd.to_datetime(values, errors="coerce")
        scope_mask = ~empty
        error_mask = scope_mask & converted.isna()
        error_type = "有效性"

    elif check == "数值":
        converted = pd.to_numeric(values, errors="coerce")
        scope_mask = ~empty
        error_mask = scope_mask & converted.isna()
        error_type = "有效性"

    elif check == "整数":
        converted = pd.to_numeric(values, errors="coerce")
        scope_mask = ~empty
        error_mask = scope_mask & (converted.isna() | (converted % 1 != 0))
        error_type = "有效性"

    elif check == "范围":
        converted = pd.to_numeric(values, errors="coerce")
        minimum = rule["最小值"]
        maximum = rule["最大值"]
        scope_mask = converted.notna()
        error_mask = scope_mask & ((converted < minimum) | (converted > maximum))
        error_type = "有效性"

    elif check == "允许值":
        allowed = str(rule["允许值"]).split("|")
        scope_mask = ~empty
        error_mask = scope_mask & ~values.isin(allowed)
        error_type = "有效性"

    elif check == "格式":
        pattern = rule["正则表达式"]
        scope_mask = ~empty
        error_mask = scope_mask & ~values.astype(str).str.match(pattern)
        error_type = "有效性"

    # 下面的规则需要另一个字段作为判断条件。
    elif check in ["不早于", "不晚于", "相同", "条件必填", "至少一个非空"]:
        other_column = rule["对照字段"]

        if other_column not in table.columns:
            rule_id = rule["规则编号"]
            raise ValueError(f"规则 {rule_id} 的对照字段不存在：{other_column}")

        other_values = table[other_column]
        other_empty = other_values.isna() | (other_values.astype("string").str.strip() == "")

        if check == "不早于":
            first_date = pd.to_datetime(values, errors="coerce")
            second_date = pd.to_datetime(other_values, errors="coerce")
            # 两个日期都能解析时才比较先后顺序。
            scope_mask = first_date.notna() & second_date.notna()
            error_mask = scope_mask & (first_date < second_date)
            error_type = "一致性"

        elif check == "不晚于":
            first_date = pd.to_datetime(values, errors="coerce")
            second_date = pd.to_datetime(other_values, errors="coerce")
            scope_mask = first_date.notna() & second_date.notna()
            error_mask = scope_mask & (first_date > second_date)
            error_type = "一致性"

        elif check == "相同":
            scope_mask = ~empty & ~other_empty
            error_mask = scope_mask & (values != other_values)
            error_type = "一致性"

        elif check == "条件必填":
            scope_mask = ~other_empty
            error_mask = scope_mask & empty
            error_type = "一致性"

        elif check == "至少一个非空":
            scope_mask = all_records
            error_mask = empty & other_empty
            error_type = "完整性"

    else:
        raise ValueError(f"没有这种检查方式：{check}")

    error_reason = rule["规则说明"]
    return scope_mask, error_mask, error_type, error_reason


In [8]:
# 依次执行规则，并把结果整理成错误明细和汇总表。
def collect_errors(table, rules):
    """逐条执行业务规则，整理错误明细和规则统计。

    Parameters
    ----------
    table : pandas.DataFrame
        待审查的数据表。
    rules : pandas.DataFrame
        已启用的业务规则。每行规则会传给 check_rule 执行。

    Returns
    -------
    tuple of pandas.DataFrame
        第一个表为错误明细。原始记录会附加源索引、规则编号、
        错误类型、错误原因和规则依据；同一记录违反多条规则时
        会在明细中出现多行。
        第二个表只汇总发现错误的规则，包含总记录数、适用记录数、
        错误记录数以及两种错误率。
    """
    invalid_tables = []
    summary_rows = []
    total_records = len(table)

    # 每条规则单独检查，方便在汇总表中按规则统计。
    for index in rules.index:
        rule = rules.loc[index]
        result = check_rule(table, rule)
        scope_mask, error_mask, error_type, error_reason = result

        error_records = error_mask.sum()

        # 没有错误的规则不写入错误明细和汇总表。
        if error_records > 0:
            invalid = table.loc[error_mask].copy()
            invalid["源索引"] = invalid.index
            invalid["规则编号"] = rule["规则编号"]
            invalid["错误类型"] = error_type
            invalid["错误原因"] = error_reason
            invalid["规则依据"] = rule["规则依据"]
            invalid_tables.append(invalid)

            applicable_records = scope_mask.sum()
            total_error_rate = round(error_records / total_records * 100, 4)
            applicable_error_rate = round(
                error_records / applicable_records * 100,
                4
            )

            summary_rows.append({
                "规则编号": rule["规则编号"],
                "错误类型": error_type,
                "业务规则": error_reason,
                "规则依据": rule["规则依据"],
                "总记录数": total_records,
                "适用记录数": applicable_records,
                "错误记录数": error_records,
                "全表错误率_pct": total_error_rate,
                "适用范围错误率_pct": applicable_error_rate
            })

    if len(invalid_tables) == 0:
        invalid_records = table.iloc[0:0].copy()
    else:
        # 每条规则产生一张错误子表，这里合并成一张明细表。
        invalid_records = pd.concat(invalid_tables, ignore_index=True)

    error_summary = pd.DataFrame(summary_rows)
    return invalid_records, error_summary

In [9]:
# 审查当前工单表并查看各规则的错误统计。
invalid_records, error_summary = collect_errors(
    work_order, business_rules
)
error_summary

,规则编号,错误类型,业务规则,规则依据,总记录数,适用记录数,错误记录数,全表错误率_pct,适用范围错误率_pct
0,WO006,一致性,closed_date 不能早于 created_date,项目检查条件,248139,243530,30,0.0121,0.0123
1,WO016,一致性,resolution_action_updated_date 不能早于 created_date,项目检查条件,248139,247372,2445,0.9853,0.9884
2,WO020,有效性,road_ramp 非空时只能为 Roadway 或 Ramp,官方规则,248139,509,202,0.0814,39.6857


In [10]:
# 读取已审核启用的错误处理规则和字段映射。
handling_rules = pd.read_excel(
    rules_file,
    sheet_name="错误处理规则",
    keep_default_na=False
)
mapping_rules = pd.read_excel(
    rules_file,
    sheet_name="字段映射",
    keep_default_na=False
)


In [11]:
# 根据错误明细修改副本，并在同一流程中完成字段映射。
def clean_work_order(
    table,
    invalid_records,
    handling_rules,
    mapping_rules,
    business_rules
):
    """按 Excel 中已启用的规则清洗工单表。

    原表不会被修改。函数返回清洗后的副本和需要人工处理的隔离记录。
    """
    cleaned = table.copy()
    removed_rows = []
    isolated_tables = []
    handled_cells = {}

    # invalid_records 已经保存了每条业务规则对应的错误行。
    active_rules = handling_rules.loc[handling_rules["是否启用"] == "是"]

    for index in active_rules.index:
        rule = active_rules.loc[index]
        rule_id = rule["规则编号"]
        field = rule["错误字段"]
        condition = rule["处理条件"]
        action = rule["处理方式"]
        target = rule["处理字段"]
        value = rule["处理值"]

        rows = invalid_records.loc[
            invalid_records["规则编号"] == rule_id,
            "源索引"
        ].unique()
        rows = pd.Index(rows)

        # 重复主键分为整行重复和同一主键内容冲突。
        if condition == "完全重复":
            rows = rows.intersection(table.index[table.duplicated(keep="first")])

        elif condition == "主键冲突":
            conflict_rows = []
            for key, group in table.loc[rows].groupby(field, dropna=False):
                if len(group.drop_duplicates()) > 1:
                    conflict_rows.extend(group.index)
            rows = pd.Index(conflict_rows)

        # 大小写或多余空格不同的文本可以映射回合法值。
        elif condition in ["可标准化", "无法标准化"]:
            business_rule = business_rules.loc[
                business_rules["规则编号"] == rule_id
            ].iloc[0]
            allowed = str(business_rule["允许值"]).split("|")
            allowed_map = {
                " ".join(item.split()).casefold(): item
                for item in allowed
            }
            values = table.loc[rows, field]
            normalized = values.astype(str).str.split().str.join(" ").str.casefold()
            can_map = normalized.isin(allowed_map)
            rows = values.index[can_map if condition == "可标准化" else ~can_map]

        if len(rows) == 0:
            continue

        # 同一字段要求不同处理时不擅自选择，直接隔离该记录。
        if action not in ["删除记录", "去重", "隔离记录"]:
            rows_to_apply = []
            action_value = value if action in ["填充固定值", "使用对照字段"] else ""

            for row in rows:
                cell = (row, target)
                current_action = (action, action_value)

                if cell not in handled_cells:
                    handled_cells[cell] = current_action
                    rows_to_apply.append(row)
                elif handled_cells[cell] != current_action:
                    conflict = table.loc[[row]].copy()
                    conflict["隔离原因"] = f"字段 {target} 的处理方式冲突"
                    isolated_tables.append(conflict)
                    removed_rows.append(row)

            rows = pd.Index(rows_to_apply)

        if action in ["删除记录", "去重"]:
            removed_rows.extend(rows)
        elif action == "隔离记录":
            isolated = table.loc[rows].copy()
            isolated["隔离原因"] = rule["处理依据"]
            isolated_tables.append(isolated)
            removed_rows.extend(rows)
        elif action == "置为空值":
            cleaned.loc[rows, target] = pd.NA
        elif action == "填充固定值":
            cleaned.loc[rows, target] = value
        elif action == "使用对照字段":
            cleaned.loc[rows, target] = cleaned.loc[rows, value]
        elif action == "标准化映射":
            values = cleaned.loc[rows, target]
            normalized = values.astype(str).str.split().str.join(" ").str.casefold()
            cleaned.loc[rows, target] = normalized.map(allowed_map)
        elif action == "保留并标记":
            cleaned.loc[rows, "处理标记"] = rule_id

    # 字段映射按优先级执行，只填充目标字段中的空值。
    active_mappings = mapping_rules.loc[
        mapping_rules["是否启用"] == "是"
    ].sort_values("优先级")

    for index in active_mappings.index:
        rule = active_mappings.loc[index]
        source = rule["来源字段"]
        target = rule["目标字段"]
        values = cleaned[source]

        if target not in cleaned.columns:
            cleaned[target] = pd.NA

        if rule["匹配方式"] == "等于":
            mask = values == rule["来源值"]
        else:
            not_empty = values.fillna("").astype(str).str.strip() != ""
            mask = not_empty & (values != rule["来源值"])

        mask = mask & cleaned[target].isna()
        cleaned.loc[mask, target] = rule["映射值"]

    cleaned = cleaned.drop(index=pd.Index(removed_rows).unique())

    # 异常值处理完以后，再转换清洗结果的数据类型。
    date_fields = [
        "created_date", "closed_date", "due_date",
        "resolution_action_updated_date"
    ]
    for field in date_fields:
        cleaned[field] = pd.to_datetime(cleaned[field], errors="coerce")

    # 用分钟保存关闭耗时，几分钟和跨天工单都能直接统计。
    cleaned["closure_duration_minutes"] = (
        (cleaned["closed_date"] - cleaned["created_date"])
        .dt.total_seconds()
        .div(60)
        .round(2)
    )

    for field in cleaned.select_dtypes(include="object").columns:
        cleaned[field] = cleaned[field].astype("string")

    category_fields = [
        "agency", "status", "open_data_channel_type", "location_type",
        "address_type", "borough", "facility_type", "vehicle_type",
        "road_ramp", "agency_name_standard"
    ]
    for field in category_fields:
        cleaned[field] = cleaned[field].astype("category")

    if len(isolated_tables) == 0:
        isolated_records = table.iloc[0:0].copy()
    else:
        isolated_records = pd.concat(isolated_tables, ignore_index=True)

    return cleaned, isolated_records


In [12]:
# 在副本上同时完成错误处理和字段映射，df 与 work_order 保持原样。
cleaned, isolated_records = clean_work_order(
    work_order,
    invalid_records,
    handling_rules,
    mapping_rules,
    business_rules
)

cleaned.shape, isolated_records.shape


((248139, 25), (0, 23))

In [13]:
# 将清洗结果拆成主表、机构表和补充字段表。
def split_work_order(table):
    """按分析用途拆分工单表，三张表可通过 unique_key 或 agency 连接。"""
    main_fields = [
        "unique_key", "created_date", "closed_date", "agency",
        "complaint_type", "descriptor", "status",
        "resolution_action_updated_date", "open_data_channel_type",
        "location_type", "borough",
        "closure_duration_minutes"
    ]
    supplement_fields = [
        "unique_key", "agency_name", "descriptor_2", "due_date",
        "address_type", "city", "facility_type", "park_facility_name",
        "park_borough", "vehicle_type", "taxi_company_borough",
        "bridge_highway_name", "road_ramp"
    ]

    fact_work_order = table.loc[:, main_fields].copy()
    dim_agency = table.loc[
        :, ["agency", "agency_name_standard"]
    ].drop_duplicates().reset_index(drop=True)
    work_order_supplement = table.loc[:, supplement_fields].copy()

    return fact_work_order, dim_agency, work_order_supplement


fact_work_order, dim_agency, work_order_supplement = split_work_order(cleaned)

fact_work_order.shape, dim_agency.shape, work_order_supplement.shape


((248139, 12), (14, 2), (248139, 13))